In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
from matplotlib.colors import ListedColormap
from matplotlib.colors import TwoSlopeNorm
from shapely.geometry import box
from datetime import datetime, timedelta
from shapely.geometry import Polygon
from shapely.geometry import Point
import os
import regionmask
import cartopy.crs as ccrs
pd.set_option("display.max_rows", 100)

# Study Region

In [ ]:
#country boundaries
cn_bd = gpd.read_file("/home/lixinh/Study_Area/Country_Boundary_eurostat/CNTR_RG_01M_2020_4326.shp")

In [ ]:
cn_bd

In [ ]:
cn_bd["EU_STAT"].unique()

In [ ]:
#filter european countries
cn_bd_eu = cn_bd[cn_bd["EU_STAT"] == "T"]

In [ ]:
cn_bd_eu["NAME_ENGL"]  # 27 countries in EU

In [ ]:
#add 14 missing regions
#Switzerland(CH), Norway(NO), England(UK), Andorra(AD), Liechtenstein(LI), Vatican City (VA), Faroe Islands(FO), and Jan Mayen(SJ)
#Bosnia and Herzegovina(BA), Montenegro(ME), Serbia(RS), Albania(AL), North Macedonia(MK), San Marino(SM)
# ISO 3166-1 alpha-2 country codes
CH = cn_bd[cn_bd["CNTR_ID"] == "CH"]   
NO = cn_bd[cn_bd["CNTR_ID"] == "NO"]
UK = cn_bd[cn_bd["CNTR_ID"] == "UK"]
AD = cn_bd[cn_bd["CNTR_ID"] == "AD"]
LI = cn_bd[cn_bd["CNTR_ID"] == "LI"]
VA = cn_bd[cn_bd["CNTR_ID"] == "VA"]
FO = cn_bd[cn_bd["CNTR_ID"] == "FO"]
SJ = cn_bd[cn_bd["CNTR_ID"] == "SJ"]
BA = cn_bd[cn_bd["CNTR_ID"] == "BA"]
ME = cn_bd[cn_bd["CNTR_ID"] == "ME"]
RS = cn_bd[cn_bd["CNTR_ID"] == "RS"]  #inlucdes Kosovo
AL = cn_bd[cn_bd["CNTR_ID"] == "AL"]
MK = cn_bd[cn_bd["CNTR_ID"] == "MK"]
SM = cn_bd[cn_bd["CNTR_ID"] == "SM"]

In [ ]:
cn_bd_study = pd.concat([cn_bd_eu, CH, NO, UK, AD, LI, VA, FO, SJ, BA, ME, RS, AL, MK, SM], ignore_index=True)

In [ ]:
cn_bd_study

In [ ]:
cn_bd_study.crs

In [ ]:
cn_bd_study.plot(figsize = (10,10))   #We need to clip the remote islands.

In [ ]:
#EnZ
eu_enz = gpd.read_file('/net/krypton/hyclimm/data/public/geospatial/regions/Environmental_Stratification_EU/EnSv8/EnSv8/enz_v8.shp')
eu_enz = eu_enz.to_crs(epsg = 4326) #WGS84 (EPSG: 4326)
eu_enz.plot(figsize = (15,15))

In [ ]:
min_lon, min_lat, max_lon, max_lat = eu_enz.total_bounds
bounding_box = box(min_lon, min_lat, max_lon, max_lat)

In [ ]:
eu_enz.total_bounds

In [ ]:
cn_bd_study = gpd.clip(cn_bd_study, bounding_box)

In [ ]:
cn_bd_study.plot(figsize = (15,15))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1,2,figsize = (20, 20))
cn_bd_study.plot(ax = ax1)
eu_enz.plot(ax = ax2)
plt.show()

# Make a Clear-cut Eastern Boundary

In [ ]:
# add Russia, Belarus, Ukraine, Moldova, Turkey
#country boundaries
cn_bd = gpd.read_file("/home/lixinh/Study_Area/Country_Boundary_eurostat/CNTR_RG_01M_2020_4326.shp")
RU = cn_bd[cn_bd["CNTR_ID"] == "RU"]
BY = cn_bd[cn_bd["CNTR_ID"] == "BY"]
UA = cn_bd[cn_bd["CNTR_ID"] == "UA"]
MD = cn_bd[cn_bd["CNTR_ID"] == "MD"]
TR = cn_bd[cn_bd["CNTR_ID"] == "TR"]

In [ ]:
eastern_cn = pd.concat([RU, BY, UA, MD, TR], ignore_index = True)

In [ ]:
eastern_cn.plot()

In [ ]:
eastern_bbox = box(19, 33, 32, 72)

In [ ]:
eastern_cn_clipped = gpd.clip(eastern_cn, eastern_bbox)

In [ ]:
eastern_cn_clipped.plot()

In [ ]:
study_area = gpd.read_file("/home/lixinh/Study_Area/Study_Area.shp")

In [ ]:
study_area.plot()

In [ ]:
study_area_final = pd.concat([study_area, eastern_cn_clipped], ignore_index = True)

In [ ]:
study_area_final.plot()

In [ ]:
study_area_final

In [ ]:
#save
study_area_final.to_file("/home/lixinh/Study_Area/Study_Area_Final.shp")

# Exclude Cyprus

In [ ]:
study_area_final = gpd.read_file("/net/rain/hyclimm/data/projects/SynFire/Study_Area/Study_Area_Final.shp")
study_area_32E = study_area_final[study_area_final["NAME_ENGL"] != "Cyprus"].reset_index(drop = True)

In [ ]:
fig, axes = plt.subplots(1, 2)
axes = axes.flatten()

study_area_final.plot(ax = axes[0])
study_area_32E.plot(ax = axes[1])

plt.show()

In [ ]:
study_area_32E.to_file("/net/rain/hyclimm/data/projects/SynFire/Study_Area/Study_Area_32E.shp")

# Create 2D land mask based on CERRA grid in EPSG:4326 for the study domain (Lon: -12-34, Lat: 72-33)

In [ ]:
fwi = xr.open_dataarray(f"/net/rain/hyclimm/data/projects/fofix/standardize_Cerra_empirically_original_res_from_shell/fwi_cerra_grid_2001-01-01_2020-12-31.nc", decode_coords = "all").sel(time = "2001-08-01")   # as a reference for CERRA grid

## Landmask for each country

In [ ]:
# define CERRA projection
proj_cerra = ccrs.LambertConformal(central_longitude=8,
                                   central_latitude=50,       
                                   false_easting=2937000.506058639,        
                                   false_northing=2937000.470434457,        
                                   standard_parallels=(50, 50))     # "+proj=lcc +lat_1=50 +lat_2=50 +lon_0=8 +lat_0=50 +x_0=2937000.506058639 +y_0=2937000.470434457 +datum=WGS84"

fwi = fwi.rio.write_crs(proj_cerra)
fwi_lonlat = fwi.rio.reproject("EPSG:4326")
fwi_lonlat = fwi_lonlat.sel(x = slice(-12, 34), y = slice(72, 33))

In [ ]:
study_area_32E = gpd.read_file("/net/rain/hyclimm/data/projects/SynFire/Study_Area/Study_Area_32E.shp")

In [ ]:
lon = fwi_lonlat.x.values
lat = fwi_lonlat.y.values

In [ ]:
land_mask = regionmask.mask_geopandas(study_area_32E, lon, lat)   # create land mask from geodataframe

In [ ]:
land_mask = land_mask.rename({"lat": "y", 
                              "lon": "x"})

In [ ]:
land_mask

In [ ]:
# all countries
f, ax = plt.subplots(subplot_kw=dict(projection=ccrs.PlateCarree()))
land_mask.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    add_colorbar=False,
)
ax.coastlines(color="0.1")

In [ ]:
# one country
f, ax = plt.subplots(subplot_kw=dict(projection=ccrs.PlateCarree()))
land_mask.where(land_mask == 4).plot(   # subset country based on the index in the study_area_32E geodataframe
    ax=ax,
    transform=ccrs.PlateCarree(),
    add_colorbar=False,
)
ax.coastlines(color="0.1")

In [ ]:
land_mask.to_netcdf("/net/rain/hyclimm/data/projects/SynFire/Study_Area/Land_Mask_Country_CERRA_reproject_EPSG4326_study_area_32E.nc")

## Binary land mask for the land (1)

In [ ]:
land_mask_binary = xr.DataArray(data = np.where(land_mask.isnull(), 0, 1),
                                coords = dict(y = land_mask.y, x = land_mask.x))

In [ ]:
f, ax = plt.subplots(subplot_kw=dict(projection=ccrs.PlateCarree()))

land_mask_binary.where(land_mask_binary == 1).plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    add_colorbar=False,
)
ax.coastlines(color="0.1")

In [ ]:
f, ax = plt.subplots(subplot_kw=dict(projection=ccrs.PlateCarree()))

land_mask_binary.where(land_mask_binary == 0).plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    add_colorbar=False,
)
ax.coastlines(color="0.1")

In [ ]:
land_mask_binary.to_netcdf("/net/rain/hyclimm/data/projects/SynFire/Study_Area/Land_Mask_Binary_CERRA_reproject_EPSG4326_study_area_32E.nc")